In [1]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np
from scipy.stats.mstats import winsorize

from utils import convert_data, winsorize_cols, run_regression, difference_test_table
from col import cols, win_cols, dep_var, main_vars, control_vars, fe_vars

In [2]:
# 轉換資料
df = pd.read_excel('0428持有公司主表_ESG已併入.xlsx')
df = convert_data(df)

In [3]:
# describe statistic
# ===== Winsorize 前的敘述性統計 =====
print("=== Winsorize 前 ===")
display(df[cols].describe(percentiles=[0.01, 0.25, 0.5, 0.75, 0.99]).T)

# ===== Winsorize 後的敘述性統計 =====
df_win = winsorize_cols(df, win_cols, limits=(0.01, 0.01))
print("=== Winsorize 後 ===")
display(df_win[cols].describe(percentiles=[0.01, 0.25, 0.5, 0.75, 0.99]).T)

=== Winsorize 前 ===


,count,mean,std,min,1%,25%,50%,75%,99%,max
Fund_id,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Year_month_lag1,8730.0,168.470724,0.104755,168.166667,168.166667,168.416667,168.500000,168.583333,168.583333,168.583333
Firm_id,8826.0,4012.390097,2182.707957,1101.000000,1134.250000,2377.000000,3044.000000,5871.000000,9938.000000,9958.000000
TSE,8826.0,22.310673,7.920598,1.000000,1.250000,20.000000,24.000000,27.000000,37.000000,38.000000
Year,8826.0,2022.637435,1.263198,2019.000000,2019.000000,2022.000000,2023.000000,2024.000000,2024.000000,2024.000000
...,...,...,...,...,...,...,...,...,...,...
Tangibility_r,8822.0,0.222114,0.166803,0.000062,0.004761,0.077470,0.200893,0.342199,0.614691,0.890922
Board_Independ_r,8820.0,0.382395,0.097030,0.000000,0.200000,0.333333,0.363636,0.428571,0.625000,0.666667
Age,8822.0,33.134890,14.997802,1.000000,5.000000,23.000000,31.000000,42.000000,72.000000,78.000000
l_Age,8822.0,3.384435,0.520669,0.000000,1.609438,3.135494,3.433987,3.737670,4.276666,4.356709


=== Winsorize 後 ===


,count,mean,std,min,1%,25%,50%,75%,99%,max
Fund_id,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Year_month_lag1,8730.0,168.470724,0.104755,168.166667,168.166667,168.416667,168.500000,168.583333,168.583333,168.583333
Firm_id,8826.0,4012.390097,2182.707957,1101.000000,1134.250000,2377.000000,3044.000000,5871.000000,9938.000000,9958.000000
TSE,8826.0,22.310673,7.920598,1.000000,1.250000,20.000000,24.000000,27.000000,37.000000,38.000000
Year,8826.0,2022.637435,1.263198,2019.000000,2019.000000,2022.000000,2023.000000,2024.000000,2024.000000,2024.000000
...,...,...,...,...,...,...,...,...,...,...
Tangibility_r,8822.0,0.221544,0.165108,0.004761,0.004761,0.077470,0.200893,0.342199,0.614691,0.614691
Board_Independ_r,8820.0,0.382405,0.096617,0.200000,0.200000,0.333333,0.363636,0.428571,0.625000,0.625000
Age,8822.0,33.134890,14.997802,1.000000,5.000000,23.000000,31.000000,42.000000,72.000000,78.000000
l_Age,8822.0,3.389659,0.496102,1.609438,1.609438,3.135494,3.433987,3.737670,4.276666,4.276666


In [4]:
# 預測哪些公司容易被議合
results = {}
for var in main_vars:
    results[var] = run_regression(df_win, y='engagement_t', x_vars=[var] + control_vars, cluster_var='Firm_id', model_type='probit')

results['all'] = run_regression(df_win, y='engagement_t', x_vars=main_vars + control_vars, cluster_var='Firm_id', model_type='probit')

for var in results.keys():
    print('================')
    print(results[var].summary())
    print('\n')

                          Probit Regression Results                           
Dep. Variable:           engagement_t   No. Observations:                 8164
Model:                         Probit   Df Residuals:                     8153
Method:                           MLE   Df Model:                           10
Date:                Thu, 30 Apr 2026   Pseudo R-squ.:                0.007470
Time:                        00:22:51   Log-Likelihood:                -2981.7
converged:                       True   LL-Null:                       -3004.2
Covariance Type:              cluster   LLR p-value:                 2.285e-06
                       coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------
Intercept           -2.0602      0.410     -5.021      0.000      -2.864      -1.256
Conglo              -0.0286      0.070     -0.408      0.684      -0.166       0.109
l_Size               0.0537 

In [5]:
table1 = difference_test_table(
    df_win, main_vars + control_vars,
    groups={
        '有議合': df_win['engagement_t'] == 1,
        '無議合': df_win['engagement_t'] == 0,
    },
    diffs=[('有議合', '無議合')],
)

table2 = difference_test_table(
    df_win, main_vars + control_vars,
    groups={
        '持有並議合': (df_win['hold_lag1'] == 1) & (df_win['engagement_t'] == 1),
        '未持有議合': (df_win['hold_lag1'] == 0) & (df_win['engagement_t'] == 1),
        '持有未議合': (df_win['hold_lag1'] == 1) & (df_win['engagement_t'] == 0),
    },
    diffs=[('持有並議合', '持有未議合'), ('持有並議合', '未持有議合')],
)

display(table1)
display(table2)

有議合                   無議合                有議合-無議合  \
                          mean     median       mean     median        mean   
Conglo                0.904858   1.000000   0.897982   1.000000      0.0069   
Seg                   3.152834   3.000000   3.120390   3.000000      0.0324   
Comp                  0.317041   0.334419   0.313876   0.334040      0.0032   
Foreign_Dummy         0.134615   0.000000   0.134928   0.000000     -0.0003   
Foreign_Sales_Ratio   0.657995   0.800292   0.650740   0.800354      0.0073   
l_Size               18.148927  17.877241  17.885075  17.592176   0.2639***   
Lev                   0.478799   0.465452   0.464324   0.447405    0.0145**   
Cash_r                0.194913   0.167318   0.201576   0.176412     -0.0067   
ROA                   9.606378   8.240000  10.060601   8.660000     -0.4542   
Current_Ratio         2.186797   1.827810   2.375122   1.933162  -0.1883***   
Tangibility_r         0.226390   0.209818   0.220870   0.197550      0.0055   
Board_Independ_r      0.379617   0.363636   0.382796   0.363636     -0.0032   
l_Age                 3.413035   3.465736   3.386400   3.433987      0.0266   
SalesGrowth           0.083537   0.064351   0.100952   0.073238   -0.0174**   

                                 
                         median  
Conglo                   0.0000  
Seg                      0.0000  
Comp                     0.0004  
Foreign_Dummy            0.0000  
Foreign_Sales_Ratio     -0.0001  
l_Size                0.2851***  
Lev                   0.0180***  
Cash_r                  -0.0091  
ROA                   -0.4200**  
Current_Ratio        -0.1054***  
Tangibility_r            0.0123  
Board_Independ_r         0.0000  
l_Age                  0.0317**  
SalesGrowth             -0.0089

持有並議合                 未持有議合                 持有未議合  \
                          mean     median       mean     median       mean   
Conglo                0.942065   1.000000   0.879865   1.000000   0.897982   
Seg                   3.443325   3.000000   2.957699   3.000000   3.120390   
Comp                  0.344625   0.395605   0.298512   0.294333   0.313876   
Foreign_Dummy         0.141058   0.000000   0.130288   0.000000   0.134928   
Foreign_Sales_Ratio   0.666749   0.814021   0.652088   0.797459   0.650740   
l_Size               18.577916  18.426130  17.860434  17.511773  17.885075   
Lev                   0.472753   0.459139   0.482865   0.470812   0.464324   
Cash_r                0.216035   0.192797   0.180709   0.154952   0.201576   
ROA                  11.631064   9.210000   8.244785   7.110000  10.060601   
Current_Ratio         2.255403   1.923397   2.140791   1.771569   2.375122   
Tangibility_r         0.212782   0.197419   0.235541   0.218766   0.220870   
Board_Independ_r      0.378491   0.363636   0.380376   0.363636   0.382796   
l_Age                 3.427741   3.496508   3.403146   3.433987   3.386400   
SalesGrowth           0.100521   0.083693   0.072125   0.058585   0.100952   

                               持有並議合-持有未議合            持有並議合-未持有議合             
                        median        mean     median        mean     median  
Conglo                1.000000   0.0441***     0.0000   0.0622***    0.0000*  
Seg                   3.000000   0.3229***  0.0000***   0.4856***  0.0000***  
Comp                  0.334040    0.0307**   0.0616**   0.0461***  0.1013***  
Foreign_Dummy         0.000000      0.0061     0.0000      0.0108     0.0000  
Foreign_Sales_Ratio   0.800354      0.0160     0.0137      0.0147     0.0166  
l_Size               17.592176   0.6928***  0.8340***   0.7175***  0.9144***  
Lev                   0.447405      0.0084     0.0117     -0.0101    -0.0117  
Cash_r                0.176412    0.0145**   0.0164**   0.0353***  0.0378***  
ROA                   8.660000   1.5705***  0.5500***   3.3863***  2.1000***  
Current_Ratio         1.933162    -0.1197*    -0.0098      0.1146   0.1518**  
Tangibility_r         0.197550     -0.0081    -0.0001   -0.0228**  -0.0213**  
Board_Independ_r      0.363636     -0.0043     0.0000     -0.0019     0.0000  
l_Age                 3.433987      0.0413    0.0625*      0.0246     0.0625  
SalesGrowth           0.073238     -0.0004     0.0105     0.0284*    0.0251*

In [8]:
def result_to_df(res):
    return pd.DataFrame({
        'coef': res.params,
        'std_err': res.bse,
        'z/t': res.tvalues,
        'p_value': res.pvalues,
    })

with pd.ExcelWriter('結果彙整.xlsx', engine='openpyxl') as writer:
    df[cols].describe(percentiles=[0.01, 0.25, 0.5, 0.75, 0.99]).T.to_excel(writer, sheet_name='Winsorize前_敘述統計')
    df_win[cols].describe(percentiles=[0.01, 0.25, 0.5, 0.75, 0.99]).T.to_excel(writer, sheet_name='Winsorize後_敘述統計')
    table1.to_excel(writer, sheet_name='差異檢定_議合')
    table2.to_excel(writer, sheet_name='差異檢定_三組')

    for name, res in results.items():
        result_to_df(res).to_excel(writer, sheet_name=f'Probit_{name}'[:31])